In [394]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "ngsolve"
solver2 = "comsol"

sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}.vtu")
sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")

In [395]:
tree = KDTree(sol_ngsolve.points)
distances, indices = tree.query(sol_comsol.points)

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")

aligned_grid = sol_comsol.copy()


for array_name in sol_ngsolve.point_data.keys():
        data = sol_ngsolve.point_data[array_name]
        
        reordered_data = data[indices]
        
        aligned_grid.point_data[array_name] = reordered_data
        print(f"Transferred array: {array_name}")

aligned_grid.save(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")

Maximum alignment error (distance): 1.601242e-13
Transferred array: magnetic_vector_potential_nd
Transferred array: magnetic_flux_density
Transferred array: current_density
Transferred array: magnetic_vector_potential
Transferred array: electric_potential


In [396]:
sol_ngsolve = pv.read(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")
elec_pot_ngsolve = sol_ngsolve["electric_potential"]
mag_flux_ngsolve = sol_ngsolve["magnetic_flux_density"]
mag_vec_ngsolve = sol_ngsolve["magnetic_vector_potential"]

sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")
elec_pot_comsol = sol_comsol["electric_potential"]
mag_flux_comsol = sol_comsol["magnetic_flux_density"]
mag_vec_comsol = sol_comsol["magnetic_vector_potential"]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_comsol.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_comsol.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_comsol.shape)

(278516, 3)
(278516, 3)
(278516, 3)
(278516, 3)


In [397]:
mesh = sol_comsol.copy()

mesh.point_data.remove("magnetic_vector_potential")
mesh.point_data.remove("electric_potential")
mesh.point_data.remove("magnetic_flux_density")

In [398]:
# print(f"Electric potential errors between {solver1} and {solver2}:")

# mesh = error(sol=elec_pot_ngsolve, sol_ref=elec_pot_comsol, 
#              eps = 1e-6, mesh=mesh, tag="scalar", save_tag="V")

In [399]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="A")

Magnetic flux density errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 1.039e-01.
  * Avg. absolute error in x direction  : 8.271e-03.

  * Max. relative error in x direction : 9.594e+05 %.
  * Avg. relative error in x direction : 1.843e+02 %.

  * Max. absolute error in y direction  : 1.001e-01.
  * Avg. absolute error in y direction  : 5.576e-03.

  * Max. relative error in y direction : 7.912e+05 %.
  * Avg. relative error in y direction : 2.805e+02 %.

  * Max. absolute error in z direction  : 1.377e-01.
  * Avg. absolute error in z direction  : 1.218e-02.

  * Max. relative error in z direction : 1.971e+06 %.
  * Avg. relative error in z direction : 2.569e+02 %.


In [400]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="B")

Magnetic vector potential errors between ngsolve and comsol:

  * Max. absolute error in x direction  : 1.491e-02.
  * Avg. absolute error in x direction  : 2.279e-04.

  * Max. relative error in x direction : 3.055e+05 %.
  * Avg. relative error in x direction : 2.830e+02 %.

  * Max. absolute error in y direction  : 2.163e-02.
  * Avg. absolute error in y direction  : 3.058e-04.

  * Max. relative error in y direction : 5.940e+04 %.
  * Avg. relative error in y direction : 1.634e+02 %.

  * Max. absolute error in z direction  : 1.943e-02.
  * Avg. absolute error in z direction  : 2.056e-04.

  * Max. relative error in z direction : 3.480e+05 %.
  * Avg. relative error in z direction : 3.940e+02 %.


In [401]:
mesh.save(f"../../output/{case_name}/{case_name}_error_ngsolve_comsol.vtu")